# Hooks Intercept: Writable Fields That Control Agent Behavior

Most hook event attributes are read-only. A small set of fields are writable by design and change agent behavior when set. This notebook demonstrates the four most useful ones:

- **`cancel_tool`** on `BeforeToolCallEvent` - block a tool and return a message to the model.
- **`retry`** on `AfterToolCallEvent` - re-run a failed tool call.
- **`resume`** on `AfterInvocationEvent` - auto-continue the agent with a follow-up prompt.
- **`messages`** on `BeforeInvocationEvent` - redact or rewrite input before the model sees it.

Every example is self-contained and runnable.

> **Previous:** [01_hooks_basics.ipynb](./01_hooks_basics.ipynb) covers registration and the event lifecycle.  
> **Next:** [03_hooks_packaging.ipynb](./03_hooks_packaging.ipynb) covers cross-hook data sharing and bundling hooks into reusable providers.


## Writable fields reference

| Field | Event | Effect |
|-------|-------|--------|
| `cancel_tool` | `BeforeToolCallEvent` | Skip the tool; return the given string to the model as the tool result. |
| `selected_tool` / `tool_use` | `BeforeToolCallEvent` | Swap the tool or edit its arguments before execution. |
| `result` | `AfterToolCallEvent` | Rewrite the tool result the model will see. |
| `retry` | `AfterToolCallEvent` | Re-invoke the same tool call with the same `tool_use_id`. |
| `retry` | `AfterModelCallEvent` | Re-invoke the model with the same inputs. |
| `resume` | `AfterInvocationEvent` | Auto-start a new agent invocation with the given input. |


## Setup

This tutorial uses Claude Haiku 4.5 on Amazon Bedrock by default. It works with any Strands-supported model. Swap the `model=` argument in the `Agent(...)` constructor if you prefer a different provider (see [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/)).


In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import logging
import re

from strands import Agent, tool
from strands.hooks import (
    BeforeInvocationEvent,
    AfterInvocationEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    HookProvider,
    HookRegistry,
)

# Use the same Bedrock inference profile as the other 01-learn tutorials.
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Quiet SDK logs so the hook demonstrations stay the focus of the output.
logging.getLogger("strands").setLevel(logging.WARNING)

## 1. `cancel_tool`: block a tool with a message to the model

Set `event.cancel_tool = "...message..."` inside a `BeforeToolCallEvent` callback. The tool never runs; the string you set becomes the tool result the model sees.


In [ ]:
@tool
def send_email(to: str, body: str) -> str:
    """Send an email.

    Args:
        to: Recipient address.
        body: Email body.
    """
    return f"Email sent to {to}"

def block_external_email(event: BeforeToolCallEvent) -> None:
    if event.tool_use["name"] == "send_email":
        recipient = event.tool_use["input"].get("to", "")
        if not recipient.endswith("@example.com"):
            event.cancel_tool = (
                f"Policy: cannot email external address {recipient!r}. "
                "Tell the user you can only email @example.com addresses."
            )

agent = Agent(model=MODEL_ID, tools=[send_email], callback_handler=None)
agent.add_hook(block_external_email)

print(agent("Email alice@gmail.com saying hi."))

The model receives our policy message as the tool result and responds by explaining the restriction. The external email was never actually sent.


## 2. `retry` on `AfterToolCallEvent`: re-run a flaky tool

Setting `event.retry = True` in an `AfterToolCallEvent` callback causes the tool executor to discard the result and re-invoke the tool with the same `tool_use_id`. Always pair this with a counter to bound retries.


In [ ]:
class FlakyCounter:
    """Deterministic flakiness: fail the first N calls, succeed afterwards."""
    def __init__(self, fail_first: int = 1):
        self.calls = 0
        self.fail_first = fail_first

counter = FlakyCounter(fail_first=1)

@tool
def lookup_stock(symbol: str) -> str:
    """Look up a stock price.

    Args:
        symbol: Ticker symbol.
    """
    counter.calls += 1
    if counter.calls <= counter.fail_first:
        raise RuntimeError("transient upstream error")
    return f"{symbol} is $142.10"

class RetryOnToolError(HookProvider):
    def __init__(self, max_retries: int = 2):
        self.max_retries = max_retries
        self.attempts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self._maybe_retry)

    def _maybe_retry(self, event: AfterToolCallEvent) -> None:
        tool_use_id = str(event.tool_use.get("toolUseId", ""))
        self.attempts[tool_use_id] = self.attempts.get(tool_use_id, 0) + 1
        if event.result.get("status") == "error" and self.attempts[tool_use_id] <= self.max_retries:
            print(f"  retrying tool (attempt {self.attempts[tool_use_id]})")
            event.retry = True

agent = Agent(model=MODEL_ID, tools=[lookup_stock], hooks=[RetryOnToolError()], callback_handler=None)
print(agent("What is the price of ACME?"))
print("tool was invoked", counter.calls, "times")

The first call raised; the `AfterToolCallEvent` hook flipped `retry = True`; the tool executor re-ran the tool with the same arguments; the retry succeeded and the model continued normally. Only the final (successful) result is added to conversation history.

The same pattern works for `AfterModelCallEvent.retry`: inspect `event.exception` and set `event.retry = True` to re-run the model call (useful for handling throttling or transient model errors).


**Two retry caveats to know:**

* **Streaming:** intermediate streaming events (`ToolStreamEvent`, model deltas) from the discarded attempt have **already** been emitted to any caller consuming `stream_async`. Only the final attempt's result is added to conversation history. Callers processing streamed events should be idempotent or track the current attempt.
* **`structured_output`:** `BeforeModelCallEvent` / `AfterModelCallEvent` (and thus model-level `retry`) do **not** fire for `Agent.structured_output(...)`. Use `BeforeInvocationEvent` / `AfterInvocationEvent` if you need to hook that code path.


## 3. `resume` on `AfterInvocationEvent`: auto-continue the agent

`AfterInvocationEvent.resume` lets a hook chain a follow-up invocation automatically. Set it to any valid agent input (string, message list, interrupt responses). Guard it with a termination condition to avoid infinite loops.


In [ ]:
summarize_done = {"done": False}

async def summarize_once(event: AfterInvocationEvent) -> None:
    if not summarize_done["done"] and event.result and event.result.stop_reason == "end_turn":
        summarize_done["done"] = True
        event.resume = "Now summarize what you just said in exactly five words."

agent = Agent(model=MODEL_ID, callback_handler=None)
agent.add_hook(summarize_once)

result = agent("List three benefits of typed hook systems.")
print("final answer:", str(result).strip())

The agent runs twice inside a single top-level `agent(...)` call: once to produce the list, and a second time (triggered by `resume`) to produce the five-word summary. A fresh `BeforeInvocationEvent` fires for the second pass, so the lifecycle is fully re-entered.


## 4. `BeforeInvocationEvent.messages`: redact input before the model sees it

`BeforeInvocationEvent` exposes a writable `messages` field. Mutating it rewrites what the agent processes for this request, without touching any previous conversation history. This is the clean place to strip PII, redact secrets, or normalize user input.


In [ ]:
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

def redact_emails(event: BeforeInvocationEvent) -> None:
    if not event.messages:
        return
    for msg in event.messages:
        for block in msg.get("content", []):
            if "text" in block:
                block["text"] = EMAIL_RE.sub("[REDACTED_EMAIL]", block["text"])

agent = Agent(model=MODEL_ID, callback_handler=None)
agent.add_hook(redact_emails)

print(agent("Did you get a message from alice@example.com this morning?"))

The model never sees the literal email address; it sees `[REDACTED_EMAIL]`. The same pattern works for secrets, account numbers, or any content you would rather not send to the model.


## Recap

You now know how to use the writable fields that let hooks control agent behavior:

- `cancel_tool` to block a tool and return a policy message.
- `retry` to re-run a failed tool (or model) call with bounded retries.
- `resume` to auto-chain a follow-up invocation.
- `messages` to redact or rewrite input before the model processes it.

### Next

- [**03_hooks_packaging.ipynb**](./03_hooks_packaging.ipynb) - share data across hooks with `invocation_state` and bundle multiple hooks into a reusable `HookProvider`.

### See also

- [**`13-human-in-the-loop`**](../13-human-in-the-loop/) - `BeforeToolCallEvent` is also `_Interruptible`: hooks can raise interrupts that pause the agent for human input.
